# 🌐 Week 4 — OpenRouter Multi-Model Control
> **Goal:** Route queries to different LLMs via one API — cheap models for simple tasks, smart models for hard ones.

---
### What you will learn
1. OpenRouter API setup
2. Calling OpenRouter with the openai SDK (it is compatible)
3. OpenRouter via PydanticAI
4. Listing & comparing models
5. Smart routing logic
6. Fallback system (if model A fails → try model B)
7. Mini-project: **Smart Router Bot**

## 1 — Setup

In [1]:
%pip install openai pydantic-ai python-dotenv httpx rich --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\Nidhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "Set OPENROUTER_API_KEY in .env!"
print("OpenRouter key loaded.")

OpenRouter key loaded.


## 2 — OpenRouter via openai SDK

In [ ]:
import httpx
from openai import OpenAI

# Step 1: fetch currently available free models from OpenRouter
models_resp = httpx.get(
    "https://openrouter.ai/api/v1/models",
    headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
    timeout=15,
)
all_models = models_resp.json().get("data", [])

# Filter: prompt price == "0" means free
free_models = [
    m["id"] for m in all_models
    if str(m.get("pricing", {}).get("prompt", "1")) == "0"
]
print(f"Found {len(free_models)} free models. Using: {free_models[0]}")

# Step 2: use the first available free model
client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")

response = client.chat.completions.create(
    model=free_models[0],
    messages=[{"role": "user", "content": "Say hello in 3 languages."}],
)
print(response.choices[0].message.content)

## 3 — OpenRouter model catalogue (cost & capability)

In [ ]:
import httpx
from rich.table import Table
from rich.console import Console

# Live fetch free models from OpenRouter
models_resp = httpx.get(
    "https://openrouter.ai/api/v1/models",
    headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
    timeout=15,
)
all_models = models_resp.json().get("data", [])

free_models = [
    m for m in all_models
    if str(m.get("pricing", {}).get("prompt", "1")) == "0"
]

# Store IDs for use in later cells
FREE_MODEL_IDS = [m["id"] for m in free_models]
print(f"Total free models available: {len(FREE_MODEL_IDS)}")

table = Table(title="Available Free Models on OpenRouter")
table.add_column("Model ID", style="cyan")
table.add_column("Context", style="yellow")

for m in free_models[:10]:  # show top 10
    ctx = str(m.get("context_length", "?"))
    table.add_row(m["id"], ctx)

Console().print(table)

## 4 — OpenRouter via PydanticAI

In [ ]:
from pydantic import BaseModel, Field
from typing import List
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

openrouter_provider = OpenAIProvider(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

def make_openrouter_model(model_id: str) -> OpenAIChatModel:
    return OpenAIChatModel(model_id, provider=openrouter_provider)

class Summary(BaseModel):
    key_points: List[str] = Field(min_length=2, max_length=5)
    one_liner: str

summary_agent = Agent(
    model=make_openrouter_model("deepseek/deepseek-chat"),
    output_type=Summary,
    system_prompt=(
        "Summarize the given text into structured key points. "
        "Respond ONLY with valid JSON — no markdown, no extra text. "
        'Format: {"key_points": ["point 1", "point 2"], "one_liner": "one sentence summary"}'
    ),
    retries=3,
)

text = """
Machine learning is a subset of artificial intelligence that allows systems
to learn and improve from experience without being explicitly programmed.
It focuses on developing computer programs that can access data and use it
to learn for themselves. The process begins with observations or data,
such as examples, direct experience, or instruction.
"""

result = await summary_agent.run(f"Summarize: {text}")
summary: Summary = result.output
print(f"One-liner: {summary.one_liner}")
for p in summary.key_points:
    print(f"  • {p}")

## 5 — Smart routing logic

In [ ]:
from enum import Enum

class QueryComplexity(str, Enum):
    SIMPLE  = "simple"
    MEDIUM  = "medium"
    COMPLEX = "complex"

def classify_query(query: str) -> QueryComplexity:
    q = query.lower()
    if any(k in q for k in ["analyze","compare","evaluate","design","architecture","trade-off"]) or len(query.split()) > 30:
        return QueryComplexity.COMPLEX
    if any(k in q for k in ["how","why","explain","summarize","list","write"]) or len(query.split()) > 10:
        return QueryComplexity.MEDIUM
    return QueryComplexity.SIMPLE

ROUTING_MAP = {
    QueryComplexity.SIMPLE:  "deepseek/deepseek-chat",
    QueryComplexity.MEDIUM:  "deepseek/deepseek-chat",
    QueryComplexity.COMPLEX: "deepseek/deepseek-chat",
}

for q in [
    "What is Python?",
    "How does garbage collection work in Python?",
    "Analyze the trade-offs between microservices and monolithic architecture for a startup.",
]:
    complexity = classify_query(q)
    print(f"Query     : {q[:60]}")
    print(f"Complexity: {complexity.value}  →  {ROUTING_MAP[complexity]}\n")

## 6 — Fallback system

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

async def call_with_fallback(query: str, primary_model: str, fallback_models: list[str]) -> str:
    provider = OpenAIProvider(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
    for model_id in [primary_model] + fallback_models:
        try:
            print(f"  Trying: {model_id}")
            result = await Agent(
                model=OpenAIChatModel(model_id, provider=provider),
                system_prompt="Answer concisely.",
            ).run(query)
            print(f"  Success: {model_id}")
            return result.output
        except Exception as e:
            print(f"  Failed ({model_id}): {type(e).__name__}")
    return "All models failed."

answer = await call_with_fallback(
    query="What is the capital of France?",
    primary_model="deepseek/deepseek-chat",
    fallback_models=["deepseek/deepseek-r1", "deepseek/deepseek-v3"],
)
print(f"\nAnswer: {answer}")

In [ ]:
# DIAGNOSTIC — run this first to see actual error
import httpx, os
from dotenv import load_dotenv
load_dotenv()
key = os.getenv("OPENROUTER_API_KEY")

resp = httpx.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
    json={"model": "deepseek/deepseek-chat", "messages": [{"role": "user", "content": "hi"}]},
    timeout=30,
)
print(f"Status: {resp.status_code}")
print(resp.text[:500])

## 🏆 Mini-Project: Smart Router Bot

In [ ]:
import os
from enum import Enum
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from dotenv import load_dotenv

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "Set OPENROUTER_API_KEY in .env!"

_POOL = [
    "deepseek/deepseek-chat",
    "deepseek/deepseek-r1",
    "deepseek/deepseek-v3",
]

class _C(str, Enum):
    SIMPLE = "simple"; MEDIUM = "medium"; COMPLEX = "complex"

def _classify(q: str) -> _C:
    ql = q.lower()
    if any(k in ql for k in ["analyze","compare","evaluate","design","architecture","trade-off"]) or len(q.split()) > 30:
        return _C.COMPLEX
    if any(k in ql for k in ["how","why","explain","summarize","list","write"]) or len(q.split()) > 10:
        return _C.MEDIUM
    return _C.SIMPLE

_ROUTE = {_C.SIMPLE: _POOL[0], _C.MEDIUM: _POOL[0], _C.COMPLEX: _POOL[0]}

class RouterResponse(BaseModel):
    answer: str
    model_used: str
    complexity: str

class SmartRouterBot:
    def __init__(self, api_key: str):
        self.provider = OpenAIProvider(base_url="https://openrouter.ai/api/v1", api_key=api_key)

    async def ask(self, query: str) -> RouterResponse:
        c = _classify(query)
        primary = _ROUTE[c]
        for mid in [primary] + [m for m in _POOL if m != primary]:
            try:
                r = await Agent(
                    model=OpenAIChatModel(mid, provider=self.provider),
                    system_prompt="Answer helpfully and concisely.",
                ).run(query)
                return RouterResponse(answer=r.output, model_used=mid, complexity=c.value)
            except Exception:
                continue
        return RouterResponse(answer="All models unavailable, try again later.", model_used="none", complexity=c.value)

bot = SmartRouterBot(api_key=OPENROUTER_API_KEY)

for q in [
    "What year was Python created?",
    "How does Python's GIL affect multi-threading?",
    "Analyze and compare async/await vs threading for a high-throughput API server.",
]:
    resp = await bot.ask(q)
    print(f"\nQuery [{resp.complexity.upper()}]: {q}")
    print(f"Model : {resp.model_used}")
    print(f"Answer: {resp.answer[:200]}{'...' if len(resp.answer) > 200 else ''}")
    print("-" * 60)

## Week 4 — Summary
| Topic | Key Takeaway |
|---|---|
| OpenRouter setup | `base_url="https://openrouter.ai/api/v1"` |
| PydanticAI integration | Use `OpenAIModel(model_id, openai_client=...)` |
| Cost control | Route simple queries to free/cheap models |
| Fallback system | Try models in priority order on failure |
| Smart routing | Classify complexity → pick model tier |

**Next: Week 5 — Tool Calling (give agents real superpowers)**